# Một số kiến trúc mạng nơ-ron nhân tạo (ANN) bổ sung

Trong notebook này, tôi sẽ trình bày sơ lược về một vài kiến trúc mạng nơ-ron quan trọng trong lịch sử. Những kiến trúc này hiện nay ít được sử dụng hơn so với Multilayer Perceptrons sâu (chương 10), mạng nơ-ron tích chập (chương 14), mạng nơ-ron hồi quy (chương 15), mạng attention (chương 16), autoencoders, mạng đối nghịch tạo sinh (GAN), hoặc các mô hình khuếch tán (chương 17). Tuy nhiên, chúng thường được nhắc đến trong các tài liệu và một số vẫn được dùng trong một phạm vi ứng dụng nhất định, vì vậy chúng rất đáng để tìm hiểu. Ngoài ra, chúng ta sẽ thảo luận về _deep belief nets_, vốn là những mô hình hiện đại nhất trong Deep Learning cho đến đầu những năm 2010. Chúng vẫn là đối tượng nghiên cứu tích cực, vì vậy có thể chúng sẽ trở lại mạnh mẽ trong tương lai.

## Mạng Hopfield (Hopfield Networks)
_Mạng Hopfield_ được giới thiệu lần đầu bởi W. A. Little vào năm 1974, sau đó được phổ biến bởi J. Hopfield vào năm 1982. Đây là các mạng _bộ nhớ kết hợp_ (associative memory): đầu tiên bạn dạy chúng một số mẫu, và sau đó khi chúng thấy một mẫu mới, chúng sẽ (hy vọng) xuất ra mẫu đã học gần nhất. Điều này làm cho chúng trở nên hữu ích cho việc nhận dạng ký tự: bạn huấn luyện mạng bằng cách cho nó xem các ví dụ về hình ảnh ký tự (mỗi pixel nhị phân tương ứng với một nơ-ron), và sau đó khi bạn đưa vào một hình ảnh ký tự mới, sau vài lần lặp, nó sẽ xuất ra ký tự đã học gần nhất.

Mạng Hopfield là các đồ thị kết nối đầy đủ (xem Hình 1 bên dưới); nghĩa là mỗi nơ-ron đều được kết nối với mọi nơ-ron khác. Lưu ý rằng trong sơ đồ, các hình ảnh có kích thước 6 × 6 pixel, vì vậy mạng nơ-ron bên trái nên chứa 36 nơ-ron (và 630 kết nối), nhưng để rõ ràng về mặt thị giác, một mạng nhỏ hơn nhiều đã được biểu diễn.

**Hình 1**: _Mạng Hopfield_

![Hopfield Network](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d5/Hopfield-net.svg/1200px-Hopfield-net.svg.png)

Thuật toán huấn luyện hoạt động bằng cách sử dụng quy tắc Hebb: đối với mỗi hình ảnh huấn luyện, trọng số giữa hai nơ-ron sẽ tăng lên nếu các pixel tương ứng cùng bật hoặc cùng tắt, nhưng giảm đi nếu một pixel bật và pixel kia tắt.

Để đưa một hình ảnh mới vào mạng, bạn chỉ cần kích hoạt các nơ-ron tương ứng với các pixel đang hoạt động. Mạng sau đó tính toán đầu ra của mọi nơ-ron, và điều này tạo ra một hình ảnh mới. Bạn có thể lấy hình ảnh mới này và lặp lại toàn bộ quá trình. Sau một thời gian, mạng đạt đến trạng thái ổn định. Thông thường, trạng thái này tương ứng với hình ảnh huấn luyện giống nhất với hình ảnh đầu vào.

Một hàm được gọi là _hàm năng lượng_ (energy function) được gắn liền với mạng Hopfield. Tại mỗi lần lặp, năng lượng giảm đi, vì vậy mạng được đảm bảo cuối cùng sẽ ổn định ở trạng thái năng lượng thấp. Thuật toán huấn luyện điều chỉnh các trọng số theo cách làm giảm mức năng lượng của các mẫu huấn luyện. Tuy nhiên, một số mẫu không có trong tập huấn luyện cũng kết thúc với năng lượng thấp, khiến mạng đôi khi ổn định ở một cấu hình chưa từng được học. Đây được gọi là _các mẫu giả_ (spurious patterns).

Một nhược điểm lớn khác của mạng Hopfield là chúng không mở rộng tốt – khả năng ghi nhớ của chúng xấp xỉ bằng 14% số lượng nơ-ron. Ví dụ, để phân loại hình ảnh 28 × 28 pixel, bạn sẽ cần một mạng Hopfield với 784 nơ-ron kết nối đầy đủ và 306,936 trọng số. Một mạng như vậy chỉ có thể học khoảng 110 ký tự khác nhau. Đó là quá nhiều tham số cho một bộ nhớ nhỏ như vậy.

## Máy Boltzmann (Boltzmann Machines)
_Máy Boltzmann_ được phát minh vào năm 1985 bởi Geoffrey Hinton và Terrence Sejnowski. Giống như mạng Hopfield, chúng là các ANN kết nối đầy đủ, nhưng chúng dựa trên _nơ-ron ngẫu nhiên_ (stochastic neurons): thay vì sử dụng hàm bước định mệnh để quyết định giá trị đầu ra, các nơ-ron này xuất ra 1 với một xác suất nào đó, và 0 nếu ngược lại. Hàm xác suất này dựa trên phân phối Boltzmann (dùng trong cơ học thống kê), do đó có tên gọi như vậy. Xem Phương trình 1.

**Phương trình 1**: _xác suất để một nơ-ron cụ thể xuất ra 1_
$$P(s_i^\text{next step}=1) = \sigma\left(\dfrac{\sum_{j=1}^N{w_{i,j}s_j + b_i}}{T}\right)$$

* $s_j$ là trạng thái của nơ-ron thứ $j$ (0 hoặc 1).
* $w_{i,j}$ là trọng số kết nối giữa nơ-ron thứ $i$ và $j$. Lưu ý rằng $w_{i,i}$ = 0.
* $b_i$ là số hạng bias của nơ-ron thứ $i$.
* _N_ là số lượng nơ-ron trong mạng.
* _T_ là một số gọi là _nhiệt độ_ (temperature) của mạng; nhiệt độ càng cao, đầu ra càng ngẫu nhiên.
* $\sigma$ là hàm logistic.

Các nơ-ron trong máy Boltzmann được chia thành hai nhóm: _đơn vị hiển thị_ (visible units) và _đơn vị ẩn_ (hidden units) (xem Hình 2).

**Hình 2**: Máy Boltzmann

![Boltzmann machine](https://upload.wikimedia.org/wikipedia/commons/thumb/7/7a/Boltzmann_machine.svg/1024px-Boltzmann_machine.svg.png)

Do tính chất ngẫu nhiên, máy Boltzmann sẽ không bao giờ ổn định ở một cấu hình cố định; thay vào đó, nó sẽ liên tục chuyển đổi giữa nhiều cấu hình. Nếu để nó chạy đủ lâu, xác suất quan sát một cấu hình cụ thể sẽ chỉ còn là hàm của trọng số kết nối và bias, không phụ thuộc vào cấu hình ban đầu. Khi đạt đến trạng thái này, mạng được gọi là ở trạng thái _cân bằng nhiệt_ (thermal equilibrium). Bằng cách thiết lập các tham số phù hợp, chúng ta có thể mô phỏng một loạt các phân phối xác suất. Đây là một loại _mô hình tạo sinh_ (generative model).

Huấn luyện máy Boltzmann nghĩa là tìm các tham số để mạng xấp xỉ phân phối xác suất của tập huấn luyện. Một mô hình tạo sinh như vậy có thể được sử dụng theo nhiều cách, ví dụ như để "sửa chữa" các hình ảnh bị nhiễu hoặc thiếu thông tin, hoặc dùng để phân loại.

Đáng tiếc là không có kỹ thuật hiệu quả nào để huấn luyện máy Boltzmann tổng quát. Tuy nhiên, các thuật toán khá hiệu quả đã được phát triển để huấn luyện _máy Boltzmann hạn chế_ (Restricted Boltzmann Machines - RBMs).

## Máy Boltzmann hạn chế (Restricted Boltzmann Machines)
Một RBM đơn giản là một máy Boltzmann trong đó không có kết nối giữa các đơn vị hiển thị với nhau hoặc giữa các đơn vị ẩn với nhau, chỉ có kết nối giữa đơn vị hiển thị và đơn vị ẩn. Hình 3 biểu diễn một RBM với ba đơn vị hiển thị và bốn đơn vị ẩn.

**Hình 3**: _Máy Boltzmann hạn chế (RBM)_

![RBM](https://upload.wikimedia.org/wikipedia/commons/thumb/e/e5/Restricted_Boltzmann_machine.svg/800px-Restricted_Boltzmann_machine.svg.png)

Một thuật toán huấn luyện rất hiệu quả gọi là [Contrastive Divergence](https://homl.info/135) đã được giới thiệu vào năm 2005 bởi Miguel Á. Carreira-Perpiñán và Geoffrey Hinton.

Thuật toán hoạt động như sau: đối với mỗi phiên bản huấn luyện **x**, thuật toán bắt đầu bằng cách nạp nó vào mạng bằng cách đặt trạng thái của các đơn vị hiển thị thành _x_<sub>1</sub>, _x_<sub>2</sub>, ..., _x_<sub>_n_</sub>. Sau đó, bạn tính toán trạng thái của các đơn vị ẩn bằng cách áp dụng phương trình ngẫu nhiên mô tả ở trên. Tiếp theo, bạn tính toán lại trạng thái của các đơn vị hiển thị để có vector **xʹ**, và tiếp tục tính lại trạng thái ẩn để có **hʹ**. Sau đó, bạn cập nhật trọng số theo Phương trình 2.

**Phương trình 2**: _Cập nhật trọng số Contrastive divergence_
$$w_{i,j} \leftarrow w_{i,j} + \eta(\mathbf{xh}^\intercal - \mathbf{x'h'}^\intercal)$$

Lợi ích lớn của thuật toán này là nó không yêu cầu đợi mạng đạt đến cân bằng nhiệt. Điều này làm cho nó hiệu quả hơn nhiều so với các thuật toán trước đó, và là thành phần then chốt cho sự thành công đầu tiên của Deep Learning dựa trên nhiều RBM xếp chồng lên nhau.

## Mạng niềm tin sâu (Deep Belief Nets)
Nhiều lớp RBM có thể được xếp chồng lên nhau; các đơn vị ẩn của RBM lớp thứ nhất đóng vai trò là các đơn vị hiển thị cho RBM lớp thứ hai, và cứ tiếp tục như vậy. Một chồng RBM như thế được gọi là _mạng niềm tin sâu_ (Deep Belief Net - DBN).

DBN có thể được huấn luyện từng lớp một bằng cách sử dụng Contrastive Divergence, bắt đầu từ các lớp thấp hơn và dần dần di chuyển lên các lớp trên cùng. Đây là [bài báo đột phá đã khởi đầu làn sóng Deep Learning vào năm 2006](https://homl.info/136).

Giống như RBM, DBN học cách tái tạo phân phối xác suất của đầu vào mà không cần bất kỳ sự giám sát nào. Tuy nhiên, chúng làm tốt hơn nhiều vì các lớp thấp hơn học các đặc trưng cấp thấp, trong khi các lớp cao hơn học các đặc trưng cấp cao. DBN cũng có thể được huấn luyện theo kiểu bán giám sát (semisupervised).

**Hình 4**: _Một mạng niềm tin sâu được cấu hình để học bán giám sát_

![Deep Belief Net](https://www.researchgate.net/profile/Yujun-Yang-11/publication/335805561/figure/fig1/AS:802868159152129@1568430635489/The-architecture-of-a-deep-belief-network-DBN.png)

Đầu tiên, RBM 1 được huấn luyện không giám sát. Sau đó, RBM 2 được huấn luyện dựa trên đầu ra của RBM 1. Cuối cùng, RBM 3 được huấn luyện bằng cách sử dụng các đơn vị ẩn của RBM 2 cùng với các đơn vị hiển thị bổ sung đại diện cho nhãn mục tiêu (target labels). Đây là bước có giám sát.

Một lợi ích lớn của cách tiếp cận bán giám sát này là bạn không cần nhiều dữ liệu huấn luyện có nhãn. Ngoài ra, DBN cũng có thể hoạt động ngược lại để tạo ra các thực thể mới từ nhãn lớp, một khả năng tạo sinh mạnh mẽ từng được dùng để tạo chú thích cho hình ảnh.

## Bản đồ tự tổ chức (Self-Organizing Maps)
_Bản đồ tự tổ chức_ (SOM) khá khác biệt so với tất cả các loại mạng nơ-ron khác. Chúng được sử dụng để tạo ra một biểu diễn chiều thấp của một tập dữ liệu chiều cao, thường phục vụ cho việc trực quan hóa, gom cụm hoặc phân loại. Các nơ-ron được trải ra trên một bản đồ (thường là 2D), và mỗi nơ-ron có một kết nối trọng số đến mọi đầu vào.

**Hình 5**: _Bản đồ tự tổ chức_

![Self-organizing map](https://upload.wikimedia.org/wikipedia/commons/thumb/3/3d/Self-organizing_map_cartesian.svg/1200px-Self-organizing_map_cartesian.svg.png)

Khi mạng đã được huấn luyện, một thực thể mới sẽ chỉ kích hoạt một nơ-ron duy nhất trên bản đồ: nơ-ron có vector trọng số gần nhất với vector đầu vào. Điều này làm cho SOM hữu ích cho việc trực quan hóa và nhận dạng giọng nói.

Thuật toán huấn luyện là không giám sát, hoạt động theo cơ chế cạnh tranh. Nơ-ron thắng cuộc sẽ điều chỉnh trọng số của mình và các nơ-ron lân cận để gần hơn với vector đầu vào. Quá trình này giúp các nơ-ron lân cận dần dần chuyên biệt hóa vào các đầu vào tương tự nhau.